# 📖 Notebook 1: REST API Design Principles

REST is the most widely used architecture for building web APIs. In this notebook we'll learn
the core principles by talking to a **live FastAPI server** and seeing exactly what happens.

## Learning Objectives

By the end of this notebook, you'll understand:
- How to model resources (the "nouns" of your API)
- The five main HTTP methods and when to use each one
- Three ways to pass data to an API (path, query, body)
- What HTTP status codes mean and why they matter

## 🛠️ Setup

Start the infrastructure first:

```bash
cd core-concepts/api-design
docker-compose up -d
```

This spins up three containers:

| Service    | URL                          | Purpose                            |
|------------|------------------------------|------------------------------------|
| FastAPI    | http://localhost:8000/docs    | The API we'll be calling           |
| PostgreSQL | localhost:5432               | Database behind the API            |
| Adminer    | http://localhost:8080         | Web UI to browse the database      |

**Adminer login:** System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `api_design_demo`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import requests
import psycopg2
import json

# ---------------------------------------------------------------------------
# Configuration — all cells in this notebook use these constants
# ---------------------------------------------------------------------------

BASE_URL = "http://localhost:8000"   # FastAPI server

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "api_design_demo",
    "user": "demo",
    "password": "demo"
}

def pp(response):
    """Pretty-print a requests Response as formatted JSON."""
    try:
        print(json.dumps(response.json(), indent=2))
    except Exception:
        print(response.text)

# ---------------------------------------------------------------------------
# Connection tests
# ---------------------------------------------------------------------------

# 1. Test the FastAPI server
try:
    r = requests.get(f"{BASE_URL}/health")
    print(f"✅ FastAPI server is running  — {r.json()}")
except Exception as e:
    print(f"❌ FastAPI connection failed: {e}")
    print("   Run: cd core-concepts/api-design && docker-compose up -d")

# 2. Test the PostgreSQL database
try:
    conn = psycopg2.connect(**DB_CONFIG)
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL connection failed: {e}")
    print("   Run: cd core-concepts/api-design && docker-compose up -d")

## 🌐 What is REST?

**REST** stands for **RE**presentational **S**tate **T**ransfer.  
It's not a library or a framework — it's a *set of conventions* for how clients and servers
talk to each other over HTTP.

The core idea is simple:

> **Treat everything as a resource (a noun) and use HTTP methods as the verbs.**

### 📚 Library Analogy

Think of a REST API like a **library**:

| Library concept     | REST equivalent          | Example                         |
|---------------------|--------------------------|---------------------------------|
| Books on shelves    | **Resources**            | Events, Venues, Bookings        |
| Book ISBN number    | **Resource identifier**  | `/events/42`                    |
| "I'd like to see…" | **GET** (read)           | `GET /events/42`                |
| "Please add this…" | **POST** (create)        | `POST /events`                  |
| "Replace this…"    | **PUT** (replace)        | `PUT /events/42`                |
| "Update the price" | **PATCH** (partial edit) | `PATCH /events/42`              |
| "Remove this book" | **DELETE** (remove)      | `DELETE /events/42`             |

The beauty is that **every developer already knows HTTP**, so your API feels familiar
before anyone reads a single line of documentation.

## 📦 Resource Modeling

The first step in designing a REST API is deciding **what your resources are**.

Resources are the *things* in your system. In our event-ticketing API we have:

- **Events** — concerts, conferences, comedy shows
- **Venues** — the places where events happen
- **Bookings** — a customer reserving tickets for an event

### Naming rules

| ✅ Good (nouns, plural) | ❌ Bad (verbs, actions)     |
|------------------------|--------------------------|
| `/events`              | `/getEvents`             |
| `/venues`              | `/createVenue`           |
| `/bookings`            | `/fetchAllBookings`      |

**Use plural nouns.** The HTTP method already tells us the action; the URL tells us *what* we're acting on.

Let's hit the API and see our resources:

In [ ]:
# Fetch the list of events (our main resource)
print("=" * 60)
print("GET /v2/events  —  list of events")
print("=" * 60)
r = requests.get(f"{BASE_URL}/v2/events", params={"limit": 3})
pp(r)

print()

# Fetch the list of venues (another resource)
print("=" * 60)
print("GET /v2/venues  —  list of venues")
print("=" * 60)
r = requests.get(f"{BASE_URL}/v2/venues")
pp(r)

## 🔧 HTTP Methods — Deep Dive

HTTP gives us a small set of *verbs*. Each one has a specific meaning:

| Method   | Purpose                  | Has Body? | Idempotent? |
|----------|--------------------------|-----------|-------------|
| `GET`    | Read / fetch data        | No        | ✅ Yes       |
| `POST`   | Create a new resource    | Yes       | ❌ No        |
| `PUT`    | Replace entire resource  | Yes       | ✅ Yes       |
| `PATCH`  | Update part of resource  | Yes       | ✅ Yes*      |
| `DELETE` | Remove a resource        | No        | ✅ Yes       |

*Idempotent* means calling it **twice** gives the same result as calling it once.
`POST` is the exception — each call creates a *new* resource.

Let's try each one 👇

### 📥 GET — Fetching Data

GET requests **read** data without changing anything on the server.
- `GET /v2/events` → returns a **list** of events
- `GET /v2/events/1` → returns a **single** event by its ID

In [ ]:
# GET a list of events (first 3 for brevity)
print("--- GET /v2/events (list) ---")
r = requests.get(f"{BASE_URL}/v2/events", params={"limit": 3})
print(f"Status: {r.status_code}")
for event in r.json()["events"]:
    print(f"  • [{event['id']}] {event['title']}  (${event['price']})")

print()

# GET a single event by ID
print("--- GET /v2/events/1 (single) ---")
r = requests.get(f"{BASE_URL}/v2/events/1")
print(f"Status: {r.status_code}")
event = r.json()["event"]
print(f"  Title:    {event['title']}")
print(f"  Category: {event['category']}")
print(f"  Venue:    {event['venue']['name']} ({event['venue']['city']})")

### 📤 POST — Creating a New Resource

POST sends data in the **request body** and asks the server to create something new.
The server typically responds with `201 Created` and the new resource's ID.

In [ ]:
# POST — Create a brand-new event
new_event = {
    "title": "Notebook Demo Concert",
    "description": "Created from our Jupyter notebook!",
    "venue_id": 1,
    "event_date": "2025-12-31",
    "price": 49.99,
    "total_tickets": 500,
    "category": "music"
}

print("--- POST /v2/events ---")
r = requests.post(f"{BASE_URL}/v2/events", json=new_event)
print(f"Status: {r.status_code}  (201 = Created)")
pp(r)

# Save the ID so we can use it in later cells
created_event_id = r.json()["id"]
print(f"\n💡 New event ID = {created_event_id}  (we'll use this below)")

### 🔄 PUT — Replacing an Entire Resource

PUT **replaces** the whole resource with the data you send.  
You must include *every* field — anything you leave out will be lost.

> Think of PUT like rewriting a whole page in a notebook.

In [ ]:
# PUT — Replace the event we just created with entirely new data
replaced_event = {
    "title": "Updated Demo Concert (PUT)",
    "description": "Completely replaced via PUT",
    "venue_id": 2,                        # moved to a different venue
    "event_date": "2026-01-15",
    "price": 79.99,                        # price changed
    "total_tickets": 1000,                 # more tickets
    "category": "music"
}

print(f"--- PUT /v2/events/{created_event_id} ---")
r = requests.put(f"{BASE_URL}/v2/events/{created_event_id}", json=replaced_event)
print(f"Status: {r.status_code}")
pp(r)

# Verify the change
print("\n--- Verify with GET ---")
r = requests.get(f"{BASE_URL}/v2/events/{created_event_id}")
event = r.json()["event"]
print(f"  Title: {event['title']}")
print(f"  Price: ${event['price']}")
print(f"  Venue: {event['venue']['name']}")

### ✏️ PATCH — Updating Part of a Resource

PATCH sends **only the fields you want to change**. Everything else stays the same.

> Think of PATCH like using correction tape on one word — the rest of the page is untouched.

In [ ]:
# PATCH — Update *only* the price
print(f"--- PATCH /v2/events/{created_event_id} ---")
r = requests.patch(
    f"{BASE_URL}/v2/events/{created_event_id}",
    json={"price": 99.99}   # only sending one field
)
print(f"Status: {r.status_code}")
pp(r)

# Verify — title should still be the PUT title, price should be 99.99
print("\n--- Verify with GET ---")
r = requests.get(f"{BASE_URL}/v2/events/{created_event_id}")
event = r.json()["event"]
print(f"  Title (unchanged): {event['title']}")
print(f"  Price (patched):   ${event['price']}")

### 🗑️ DELETE — Removing a Resource

DELETE removes a resource from the server.  
A successful delete usually returns `204 No Content` (empty body).

In [ ]:
# DELETE the event we created
print(f"--- DELETE /v2/events/{created_event_id} ---")
r = requests.delete(f"{BASE_URL}/v2/events/{created_event_id}")
print(f"Status: {r.status_code}  (204 = No Content — success, nothing to return)")

# Verify it's gone — should get 404 Not Found
print(f"\n--- GET /v2/events/{created_event_id}  (should be 404) ---")
r = requests.get(f"{BASE_URL}/v2/events/{created_event_id}")
print(f"Status: {r.status_code}  (404 = Not Found — the event is gone)")
pp(r)

### 📊 Method Summary

| Method   | URL example           | What it does            | Idempotent? | Safe? |
|----------|-----------------------|-------------------------|-------------|-------|
| `GET`    | `/events`             | List all events         | ✅ Yes       | ✅ Yes |
| `GET`    | `/events/42`          | Get one event           | ✅ Yes       | ✅ Yes |
| `POST`   | `/events`             | Create new event        | ❌ No        | ❌ No  |
| `PUT`    | `/events/42`          | Replace event 42        | ✅ Yes       | ❌ No  |
| `PATCH`  | `/events/42`          | Partially update 42     | ✅ Yes*      | ❌ No  |
| `DELETE` | `/events/42`          | Remove event 42         | ✅ Yes       | ❌ No  |

- **Idempotent** = calling it N times has the same effect as calling it once.  
- **Safe** = it doesn't change anything on the server (read-only).

*`PATCH` is idempotent when applied with the same data. Some patch formats (like JSON Patch with `add` operations) may not be.

## 📨 Passing Data to APIs

There are **three** places you can send data with an HTTP request:

### 1. Path Parameters — *which* resource
Built into the URL itself. Used to **identify** a specific resource.
```
GET /events/123          ← 123 is a path parameter
GET /events/123/bookings ← 123 identifies the parent event
```

### 2. Query Parameters — *how* to filter or modify
Appended after `?` in the URL. Used for **filtering**, **sorting**, **pagination**.
```
GET /events?category=music        ← filter by category
GET /events?limit=5&cursor=10     ← pagination
```

### 3. Request Body — the *data* itself
Sent as JSON in the body of the request. Used for **creating** or **updating** resources.
```json
POST /events
Body: {"title": "Jazz Night", "price": 30.00, ...}
```

Let's see all three in action:

In [ ]:
# --- 1. Path Parameter: identify a specific event ---
event_id = 1
print(f"1️⃣  Path parameter — GET /v2/events/{event_id}")
r = requests.get(f"{BASE_URL}/v2/events/{event_id}")
print(f"   → {r.json()['event']['title']}")

print()

# --- 2. Query Parameters: filter the list ---
print("2️⃣  Query parameters — GET /v2/events?category=music&limit=3")
r = requests.get(f"{BASE_URL}/v2/events", params={"category": "music", "limit": 3})
for ev in r.json()["events"]:
    print(f"   → [{ev['id']}] {ev['title']}  (category: {ev['category']})")

print()

# --- 3. Request Body: create a resource ---
body = {
    "title": "Data Passing Demo",
    "description": "Shows how request bodies work",
    "venue_id": 3,
    "event_date": "2025-09-01",
    "price": 25.00,
    "total_tickets": 200,
    "category": "tech"
}
print("3️⃣  Request body — POST /v2/events")
r = requests.post(f"{BASE_URL}/v2/events", json=body)
print(f"   → Created event ID {r.json()['id']}")

# Clean up so we don't leave junk in the database
requests.delete(f"{BASE_URL}/v2/events/{r.json()['id']}")
print("   → (cleaned up)")

## 🪆 Nested Resources

When one resource *belongs to* another, we nest the URL:

```
/events/{event_id}/bookings       ← bookings for a specific event
```

This makes the **parent-child relationship** obvious from the URL alone.

In our API:
- An **Event** can have many **Bookings**
- `POST /v2/events/1/bookings` → create a booking for event 1
- `GET  /v2/events/1/bookings` → list all bookings for event 1

In [ ]:
# First, create a fresh event so we have a clean slate
fresh_event = {
    "title": "Nested Resource Demo",
    "description": "For demonstrating bookings",
    "venue_id": 1,
    "event_date": "2025-11-01",
    "price": 50.00,
    "total_tickets": 100,
    "category": "tech"
}
r = requests.post(f"{BASE_URL}/v2/events", json=fresh_event)
eid = r.json()["id"]
print(f"Created event {eid} for our demo\n")

# --- POST a booking (nested under the event) ---
print(f"--- POST /v2/events/{eid}/bookings ---")
booking = {"user_id": 1, "quantity": 2}
r = requests.post(f"{BASE_URL}/v2/events/{eid}/bookings", json=booking)
print(f"Status: {r.status_code}")
pp(r)

print()

# --- GET all bookings for this event ---
print(f"--- GET /v2/events/{eid}/bookings ---")
r = requests.get(f"{BASE_URL}/v2/events/{eid}/bookings")
print(f"Status: {r.status_code}")
pp(r)

# Clean up
requests.delete(f"{BASE_URL}/v2/events/{eid}")

## 🚦 Status Codes

HTTP status codes are a **three-digit number** the server sends back to tell you what happened.
They are grouped by the first digit:

| Range   | Meaning               | Examples                          |
|---------|-----------------------|-----------------------------------|
| **2xx** | ✅ Success             | 200 OK, 201 Created, 204 No Content |
| **4xx** | ❌ Client error        | 400 Bad Request, 404 Not Found, 429 Too Many Requests |
| **5xx** | 💥 Server error        | 500 Internal Server Error         |

**Why do they matter?** Because code that *calls* your API reads the status code to decide
what to do next. A `201` means "great, it was created". A `404` means "that doesn't exist".
A `429` means "slow down, you're sending too many requests".

Let's trigger the most common ones:

In [ ]:
# --- 200 OK — Successful read ---
r = requests.get(f"{BASE_URL}/v2/events")
print(f"200 OK            — GET /v2/events             → {r.status_code}")

# --- 201 Created — Resource created ---
new = {
    "title": "Status Code Demo",
    "venue_id": 1,
    "event_date": "2025-10-01",
    "price": 10.00,
    "total_tickets": 100,
    "category": "tech"
}
r = requests.post(f"{BASE_URL}/v2/events", json=new)
status_demo_id = r.json()["id"]
print(f"201 Created       — POST /v2/events            → {r.status_code}")

# --- 404 Not Found — Resource doesn't exist ---
r = requests.get(f"{BASE_URL}/v2/events/99999")
print(f"404 Not Found     — GET /v2/events/99999       → {r.status_code}")

# --- 400 Bad Request — Invalid data ---
# Try booking more tickets than available (total_tickets=100, tickets_sold=0)
r = requests.post(
    f"{BASE_URL}/v2/events/{status_demo_id}/bookings",
    json={"user_id": 1, "quantity": 9999}   # way more than available!
)
print(f"400 Bad Request   — POST overbooking           → {r.status_code}  ({r.json()['detail']})")

# --- 204 No Content — Successful delete (no body returned) ---
r = requests.delete(f"{BASE_URL}/v2/events/{status_demo_id}")
print(f"204 No Content    — DELETE /v2/events/{status_demo_id}        → {r.status_code}")

## 🎯 Key Takeaways

1. **Resources are nouns, methods are verbs.**  
   Name your URLs after *things* (`/events`, `/venues`). Let the HTTP method say the action.

2. **Path params for identity, query params for filters, body for data.**  
   - `/events/42` — which event  
   - `/events?category=music` — which subset  
   - `{"title": "..." }` — the data to create or update  

3. **GET / PUT / DELETE are idempotent; POST is not.**  
   You can safely retry GET, PUT, and DELETE without side effects.
   Calling POST twice creates two resources.

4. **Use proper status codes to communicate what happened.**  
   `200` for success, `201` for created, `404` for not found, `400` for bad input.
   Your API callers depend on these to handle responses correctly.

---

**Next notebook →** We'll look at pagination, filtering, and API versioning.